# Microsoft AutoGen: Orquestación Multi-Agente

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/05-autogen.ipynb)

AutoGen permite crear sistemas de múltiples agentes que colaboran para resolver tareas complejas.

In [ ]:
!pip install pyautogen anthropic -q

In [ ]:
import os
# AutoGen puede usar múltiples backends de LLM
# Configuración para Claude
config_list_claude = [
    {
        'model': 'claude-sonnet-4-6',
        'api_key': os.environ.get('ANTHROPIC_API_KEY', 'tu-api-key'),
        'api_type': 'anthropic',
    }
]

llm_config = {
    'config_list': config_list_claude,
    'cache_seed': 42,
    'temperature': 0.1,
}

print('Configuración AutoGen lista para Claude')

## 1. ConversableAgent: agente básico

In [ ]:
from autogen import ConversableAgent, UserProxyAgent

# Agente asistente
asistente = ConversableAgent(
    name='Asistente',
    system_message='Eres un experto en Python y machine learning. Explica conceptos de forma clara.',
    llm_config=llm_config,
    human_input_mode='NEVER',
    max_consecutive_auto_reply=3,
)

# Proxy de usuario (puede ejecutar código)
usuario = UserProxyAgent(
    name='Usuario',
    human_input_mode='NEVER',
    code_execution_config={'work_dir': '/tmp/autogen_work', 'use_docker': False},
    max_consecutive_auto_reply=2,
)

# Iniciar conversación
resultado = usuario.initiate_chat(
    asistente,
    message='Escribe una función Python para calcular la distancia euclidiana entre dos vectores.',
    max_turns=2,
)

print('\nConversación completada')

## 2. GroupChat: múltiples agentes colaborando

In [ ]:
from autogen import GroupChat, GroupChatManager

# Agentes especializados
arquitecto = ConversableAgent(
    name='Arquitecto',
    system_message='Eres un arquitecto de software. Diseñas la estructura y los componentes del sistema.',
    llm_config=llm_config,
    human_input_mode='NEVER',
)

desarrollador = ConversableAgent(
    name='Desarrollador',
    system_message='Eres un desarrollador Python senior. Implementas el código siguiendo el diseño del arquitecto.',
    llm_config=llm_config,
    human_input_mode='NEVER',
)

revisor = ConversableAgent(
    name='Revisor',
    system_message='Eres un revisor de código. Identificas bugs, mejoras y problemas de seguridad.',
    llm_config=llm_config,
    human_input_mode='NEVER',
)

# Proxy de usuario como orquestador
orquestador = UserProxyAgent(
    name='Orquestador',
    human_input_mode='NEVER',
    code_execution_config=False,
)

# Configurar GroupChat
groupchat = GroupChat(
    agents=[arquitecto, desarrollador, revisor, orquestador],
    messages=[],
    max_round=6,
    speaker_selection_method='round_robin',
)

manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# Iniciar el GroupChat
orquestador.initiate_chat(
    manager,
    message='Necesitamos una función para validar emails en Python con regex. Arquitecto: diseña. Desarrollador: implementa. Revisor: revisa.',
)

print('GroupChat completado')

## 3. Agentes con herramientas personalizadas

In [ ]:
from autogen import ConversableAgent

# Definir herramienta
def calcular_estadisticas(datos: list[float]) -> dict:
    """Calcula estadísticas descriptivas de una lista de números."""
    import statistics
    return {
        'media': statistics.mean(datos),
        'mediana': statistics.median(datos),
        'desviacion': statistics.stdev(datos) if len(datos) > 1 else 0,
        'min': min(datos),
        'max': max(datos),
        'n': len(datos),
    }

# Demo sin AutoGen (la integración real de tools requiere configuración adicional)
datos_ejemplo = [23, 45, 12, 67, 34, 89, 23, 45, 56, 78]
stats = calcular_estadisticas(datos_ejemplo)
print('Estadísticas calculadas:')
for k, v in stats.items():
    print(f'  {k}: {v:.2f}' if isinstance(v, float) else f'  {k}: {v}')

## Resumen

- **ConversableAgent**: agente individual configurable
- **UserProxyAgent**: proxy humano / ejecutor de código
- **GroupChat**: varios agentes colaborando en rondas
- **GroupChatManager**: orquesta el flujo del chat
- AutoGen es ideal para pipelines donde cada agente tiene un rol especializado